# MeChess 3/3: an opening explorer from the Lichess database (CPU notebook)

**Accelerator: None.** Only an adapter around `chessme explorer-build` (also runs on a laptop). It streams a Lichess monthly database dump (CC0; the 30 GB file is never downloaded whole) and counts every position
and move of the first 24 plies for **every rating range** (under 800, then 200-point bands up to 2600, and 2600+), and writes:

| File | Content |
|---|---|
| `explorer.db` | SQLite explorer: per rating band and position, each move's **games, White / draw / Black results (the Lichess bar), average rating, average engine evaluation** (from annotated games), plus opening names, per-band population statistics and clock use per ply |
| `theory_LO_HI.bin` | one playable opening book per band (`mechess --book <folder>` picks the band of the target Elo) |
| `report.md` | coverage on held-out games, results by band, game length, time per move |

**Run:** Settings -> Internet On, Accelerator None; set `REPO_URL`; **Save Version -> Save & Run All (Commit)**. Expect roughly 2 to 5 hours with the defaults (20 million games scanned; one in four looked at in the common bands, every game in bands from 2200 up).

**Checkpointing.** A checkpoint is written every million games and every 15 minutes (atomic; refused if the settings changed). If the time budget `MAX_MINUTES` is reached, or the run is stopped, the outputs are still written from what was counted and the checkpoint is kept:
add this notebook's earlier output as an input and rerun, and it continues from the checkpoint. A finished run deletes its checkpoint. Memory is guarded (rare entries are pruned past `MAX_MEMORY_GB`). The stream reconnects itself if the connection drops.

In [ ]:
import os, subprocess, sys, pathlib

REPO_URL = "https://github.com/<you>/MeChess.git"      # <- put your repository URL here
MONTH = "2026-06"                                       # a month on https://database.lichess.org (standard rated)
MAX_SCAN = 20_000_000                                   # games to scan
SAMPLE_EVERY = 4                                        # look at one game in this many (bands from 2200 up: all games)
MAX_MINUTES = 600                                       # time budget; outputs are written even if it is reached
MAX_MEMORY_GB = 20                                      # prune the rarest entries past this
OUT = "/kaggle/working/explorer" if os.path.exists("/kaggle") else "explorer_local"

def sh(*args):
    """Run a command and stream its output; stop the notebook if it fails."""
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait():
        raise RuntimeError(f"failed: {' '.join(args)}")

if not os.path.exists("chessme"):                      # Kaggle: fetch the repository (Internet must be on)
    if "<you>" in REPO_URL:
        raise SystemExit("Set REPO_URL to your repository first.")
    sh("git", "clone", "--depth", "1", REPO_URL, "MeChess")
    os.chdir("MeChess")
sh(sys.executable, "-m", "pip", "-q", "install", "python-chess", "numpy", "pyyaml", "requests", "zstandard")
CLI = [sys.executable, "-m", "chessme"]              # every step below is one `chessme` command: the same ones you run on a laptop

SOURCE = f"https://database.lichess.org/standard/lichess_db_standard_rated_{MONTH}.pgn.zst"

## Step 1. Check: read a few hundred games (seconds). Stops here if the URL, decompression or parser is broken.

In [ ]:
sh(*CLI, "explorer-build", "--check", "--source", SOURCE)

## Step 2. Count and build (the long step; checkpointed and resumable)

In [ ]:
sh(*CLI, "explorer-build", "--source", SOURCE, "--out", OUT, "--max-scan", str(MAX_SCAN), "--sample-every", str(SAMPLE_EVERY),
   "--max-minutes", str(MAX_MINUTES), "--max-memory-gb", str(MAX_MEMORY_GB), "--checkpoint-minutes", "15",
   "--resume-glob", "/kaggle/input/**/explorer/state.pkl.gz", "--drop-state-when-finished", "--log", f"{OUT}/explorer.log")

## Step 3. Try the explorer (games, share, average rating, evaluation, and the White / draw / Black bar)

In [ ]:
AFTER_E4 = "rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR b KQkq - 0 1"
for rating in (900, 1500, 2000, 2500):
    print(f"--- rating {rating}, after 1.e4")
    sh(*CLI, "explorer-query", f"{OUT}/explorer.db", "--rating", str(rating), "--fen", AFTER_E4, "--top", "5")